# MODIS Snow Processing Workflow
This notebook preprocesses the 2024 MODIS/Terra Snow Cover Daily L3 Global 500m SIN Grid V061 data and saves clipped, cleaned snow rasters as Cloud-Optimized GeoTIFFs (COGs). It also derives summary products for your study area.


## MODIS Snow Preprocessing Workflow
This notebook preprocesses the 2024 MODIS/Terra Snow Cover Daily L3 Global 500m SIN Grid V061 data, clips it to the `aoi_gb_north_kpk_ajk.geojson` boundary, and exports Cloud-Optimized GeoTIFFs (COGs) plus derived summary products.

Run this notebook with the `sds` conda environment active / selected as the kernel.

In [3]:
from pathlib import Path

root = Path.cwd()
snow_dir = (root.parent / 'snow_data' / 'MOD10A1_61-20260412_161448').resolve()
aoi_path = (root / 'sds' / 'aoi_gb_north_kpk_ajk.geojson').resolve()
output_cog_dir = (root / 'outputs' / 'snow_cogs').resolve()
output_summary_dir = (root / 'outputs' / 'snow_summary').resolve()
output_cog_dir.mkdir(parents=True, exist_ok=True)
output_summary_dir.mkdir(parents=True, exist_ok=True)

hdf_files = sorted(snow_dir.glob('*.hdf'))
print('Notebook root:', root)
print('Snow source folder:', snow_dir)
print('AOI path:', aoi_path, 'exists:', aoi_path.exists())
print('COG output folder:', output_cog_dir)
print('Summary output folder:', output_summary_dir)
print('HDF count:', len(hdf_files))
print('Sample HDF:', hdf_files[0] if hdf_files else None)

Notebook root: c:\Users\AtifA\Desktop\Wasif SDS Project\sds
Snow source folder: C:\Users\AtifA\Desktop\Wasif SDS Project\snow_data\MOD10A1_61-20260412_161448
AOI path: C:\Users\AtifA\Desktop\Wasif SDS Project\sds\sds\aoi_gb_north_kpk_ajk.geojson exists: True
COG output folder: C:\Users\AtifA\Desktop\Wasif SDS Project\sds\outputs\snow_cogs
Summary output folder: C:\Users\AtifA\Desktop\Wasif SDS Project\sds\outputs\snow_summary
HDF count: 732
Sample HDF: C:\Users\AtifA\Desktop\Wasif SDS Project\snow_data\MOD10A1_61-20260412_161448\MOD10A1.A2024001.h23v05.061.2024004183105.hdf


In [4]:
import geopandas as gpd

aoi = gpd.read_file(aoi_path)
print('AOI CRS:', aoi.crs)
print('Number of features:', len(aoi))
print('AOI bounds:', aoi.total_bounds)

aoi = aoi.to_crs(epsg=32643)
print('AOI bounds in EPSG:32643:', aoi.total_bounds)

AOI CRS: EPSG:32643
Number of features: 1
AOI bounds: [ 131981.61696435 3625653.49548206  744155.66673087 4105739.32377728]
AOI bounds in EPSG:32643: [ 131981.61696435 3625653.49548206  744155.66673087 4105739.32377728]


In [5]:
import rasterio
from rasterio import open as rio_open

hdf_path = hdf_files[0]
print('HDF path:', hdf_path)

hdf_path_text = str(hdf_path).replace('\\', '/')
subdataset = f'HDF4_EOS:EOS_GRID:"{hdf_path_text}":MODIS_Grid_Snow_500m:Snow_Cover'
print('Trying subdataset path:', subdataset)

try:
    with rio_open(subdataset) as src:
        print('opened subdataset')
        print('count', src.count)
        print('crs', src.crs)
        print('bounds', src.bounds)
        print('dtype', src.dtypes)
        print('res', src.res)
except Exception as e:
    print('subdataset open failed:', type(e).__name__, e)
    try:
        with rio_open(hdf_path) as src:
            print('direct open succeeded unexpectedly')
            print('subdatasets', src.subdatasets)
    except Exception as e2:
        print('direct open failed:', type(e2).__name__, e2)

HDF path: C:\Users\AtifA\Desktop\Wasif SDS Project\snow_data\MOD10A1_61-20260412_161448\MOD10A1.A2024001.h23v05.061.2024004183105.hdf
Trying subdataset path: HDF4_EOS:EOS_GRID:"C:/Users/AtifA/Desktop/Wasif SDS Project/snow_data/MOD10A1_61-20260412_161448/MOD10A1.A2024001.h23v05.061.2024004183105.hdf":MODIS_Grid_Snow_500m:Snow_Cover
subdataset open failed: RasterioIOError 'HDF4_EOS:EOS_GRID:"C:/Users/AtifA/Desktop/Wasif SDS Project/snow_data/MOD10A1_61-20260412_161448/MOD10A1.A2024001.h23v05.061.2024004183105.hdf":MODIS_Grid_Snow_500m:Snow_Cover' does not exist in the file system, and is not recognized as a supported dataset name.
direct open failed: RasterioIOError 'C:\Users\AtifA\Desktop\Wasif SDS Project\snow_data\MOD10A1_61-20260412_161448\MOD10A1.A2024001.h23v05.061.2024004183105.hdf' not recognized as being in a supported file format.


In [7]:
import shutil
print('conda path', shutil.which('conda'))

conda path C:\Users\AtifA\miniconda3\Scripts\conda.EXE


In [9]:
import subprocess
cmd = ['conda', 'install', '-n', 'sds', '-c', 'conda-forge', 'gdal', 'pyhdf', '-y']
print('Installing packages:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print('returncode', result.returncode)
print(result.stdout[-4000:])
print(result.stderr[-4000:])

SyntaxError: invalid syntax (657046674.py, line 8)

In [ ]:
import numpy as np
import xarray as xr
import rioxarray
import pandas as pd
from tqdm import tqdm


def read_modis_snow(hdf_path):
    hdf_path_text = str(hdf_path).replace('\\', '/')
    subds = f'HDF4_EOS:EOS_GRID:"{hdf_path_text}":MODIS_Grid_Snow_500m:Snow_Cover'
    da = rioxarray.open_rasterio(subds, masked=True)
    return da.squeeze('band', drop=True)


def clip_to_aoi(da, aoi_gdf):
    if not aoi_gdf.crs == da.rio.crs:
        aoi_gdf = aoi_gdf.to_crs(da.rio.crs)
    return da.rio.clip(aoi_gdf.geometry, aoi_gdf.crs, drop=True, invert=False)


def write_cog(dataarray, out_path, compress='DEFLATE'):
    dataarray.rio.to_raster(
        out_path,
        driver='COG',
        compress=compress,
        tiled=True,
        blocksize=512,
        dtype=dataarray.dtype,
    )
    return out_path


def build_summary(stack_da, out_dir):
    snow_days = (stack_da == 200).sum(dim='time')
    snow_frequency = snow_days / stack_da.sizes['time']
    write_cog(snow_days, out_dir / 'snow_days.tif')
    write_cog(snow_frequency, out_dir / 'snow_frequency.tif')
    return snow_days, snow_frequency

print('Helper functions defined.')

In [ ]:
stack = []
dates = []
for hdf_file in tqdm(hdf_files, desc='Processing HDF files'):
    da = read_modis_snow(hdf_file)
    clipped = clip_to_aoi(da, aoi)
    date = hdf_file.stem.split('.')[1][1:]
    output_path = output_cog_dir / f'snow_{date}.tif'
    write_cog(clipped, output_path)
    stack.append(clipped)
    dates.append(pd.to_datetime(date, format='%Y%j'))

combined = xr.concat(stack, dim='time')
combined = combined.assign_coords(time=('time', dates))
combined.rio.write_crs(aoi.crs, inplace=True)

snow_days, snow_frequency = build_summary(combined, output_summary_dir)
summary_stats = pd.DataFrame({
    'date': combined.time.values,
    'snowy_pixels': ((combined == 200).sum(dim=['x', 'y'])).values,
})
summary_stats.to_parquet(output_summary_dir / 'snow_daily_stats.parquet')
print('Completed processing. Daily COGs and summary products are saved.')
